# Notebook 07 — Adaptive Hedge Ratio via Kalman Filter

**Adaptive Pair Trading using Cointegration, Volatility and ML Diagnostics**  
**Author:** Ayush Arora (MQMS2404)

---

## Why this notebook is the core of the project

All prior notebooks used a **static OLS hedge ratio** (β = 0.245) estimated once on the full
10-year sample. This creates two problems:

1. **Look-ahead bias**: the hedge ratio uses future data at every historical point  
2. **Structural drift**: the TATASTEEL / HINDALCO relationship shifts with commodity cycles,
   corporate actions, and sectoral rotations — a fixed β accumulates basis risk over time

The **Kalman Filter** solves both problems simultaneously.  
It estimates a **time-varying** β_t at each date using only data up to that date,
and it updates the estimate continuously as new observations arrive.

### State-Space Model

```
Observation:   A_t = α_t + β_t · B_t + ε_t,     ε_t ~ N(0, R)
State dynamics: [α_t, β_t]ᵀ = [α_{t-1}, β_{t-1}]ᵀ + w_t,   w_t ~ N(0, Q)
```

The state vector `[α_t, β_t]` follows a **random walk** — sensible for slowly-drifting
financial relationships.  δ (process noise) controls the speed of adaptation.

### Comparison
- Section A: Kalman Filter vs OLS hedge ratio over time  
- Section B: Adaptive spread vs static spread — stationarity comparison  
- Section C: Full backtest with adaptive hedge ratio

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

## Section A: Kalman Filter — Adaptive Hedge Ratio

In [ ]:
prices = pd.read_csv('data/prices.csv', index_col=0, parse_dates=True)
A = prices['TATASTEEL.NS']
B = prices['HINDALCO.NS']

# ── Static OLS benchmark ────────────────────────────────────────────────────
ols_model  = sm.OLS(A, sm.add_constant(B)).fit()
beta_ols   = ols_model.params.iloc[1]
alpha_ols  = ols_model.params.iloc[0]

print(f"Static OLS: α = {alpha_ols:.4f}, β = {beta_ols:.4f}")

In [ ]:
def kalman_hedge(y, x, delta=1e-4):
    """
    Kalman filter for time-varying cointegration coefficients.

    State:      theta_t = [alpha_t, beta_t]
    Transition: theta_t = theta_{t-1} + w_t,  w_t ~ N(0, Q)
    Obs:        y_t     = H_t @ theta_t + e_t, e_t ~ N(0, R)

    R is estimated online from the innovation variance.
    delta controls adaptiveness: larger delta → faster adaptation, noisier β.
    Typical range: 1e-5 (very slow) to 1e-3 (fast-adapting).
    """
    n      = len(y)
    theta  = np.zeros((n, 2))           # [alpha_t, beta_t]
    P      = np.eye(2) * 10.0           # initial state covariance (diffuse prior)
    Q      = delta / (1 - delta) * np.eye(2)   # process noise (constant)

    e      = np.zeros(n)                # innovations
    var_e  = np.zeros(n)                # innovation variances (R_t)
    R      = 1.0                        # initial observation noise

    for t in range(n):
        H = np.array([[1.0, x.iloc[t]]])

        # ── Predict ─────────────────────────────────────────────
        if t > 0:
            theta_pred = theta[t - 1]
        else:
            theta_pred = np.zeros(2)
        P_pred = P + Q

        # ── Innovation ──────────────────────────────────────────
        y_hat      = float(H @ theta_pred)
        e[t]       = y.iloc[t] - y_hat
        var_e[t]   = float(H @ P_pred @ H.T) + R

        # ── Update R online (exponential smoothing of squared innovations) ──
        R = 0.95 * R + 0.05 * e[t] ** 2

        # ── Kalman gain & state update ───────────────────────────
        K         = P_pred @ H.T / var_e[t]
        theta[t]  = theta_pred + K.flatten() * e[t]
        P         = (np.eye(2) - K @ H) @ P_pred

    return (
        pd.Series(theta[:, 0], index=y.index, name='kf_alpha'),
        pd.Series(theta[:, 1], index=y.index, name='kf_beta'),
        pd.Series(e,            index=y.index, name='kf_spread'),
        pd.Series(var_e,        index=y.index, name='kf_var'),
    )

kf_alpha, kf_beta, kf_spread, kf_var = kalman_hedge(A, B, delta=1e-4)

print(f"Kalman β range:  [{kf_beta.min():.4f}, {kf_beta.max():.4f}]")
print(f"Kalman β final:  {kf_beta.iloc[-1]:.4f}  (OLS static: {beta_ols:.4f})")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(kf_beta, color='crimson',   linewidth=0.9, label='Kalman β_t (adaptive)')
axes[0].axhline(beta_ols, color='navy', linewidth=1.2, linestyle='--', label=f'OLS β = {beta_ols:.4f} (static)')
axes[0].fill_between(kf_beta.index, kf_beta, beta_ols, alpha=0.08, color='crimson')
axes[0].set_ylabel('Hedge Ratio (β)')
axes[0].set_title('Kalman Filter vs OLS: Time-Varying Hedge Ratio')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(kf_alpha, color='steelblue', linewidth=0.9, label='Kalman α_t (intercept drift)')
axes[1].axhline(alpha_ols, color='navy', linewidth=1.2, linestyle='--', label=f'OLS α = {alpha_ols:.4f}')
axes[1].set_ylabel('Intercept (α)')
axes[1].set_title('Kalman Filter: Time-Varying Intercept')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Section B: Adaptive Spread vs Static Spread — Stationarity Comparison

The Kalman spread is the **innovation sequence** e_t from the filter:  
`e_t = A_t − α_t − β_t · B_t`

Because the filter continuously adjusts β_t, the innovation should be more stationary
than the residual from a fixed-β model.  We verify this with ADF tests and visual inspection.

In [ ]:
static_spread = A - beta_ols * B - alpha_ols    # OLS residual (static β)
kf_spread_s   = kf_spread                        # Kalman innovations (adaptive β)

# Drop warm-up period (first 60 rows)
static_spread = static_spread.iloc[60:]
kf_spread_s   = kf_spread_s.iloc[60:]

adf_static = adfuller(static_spread.dropna())
adf_kf     = adfuller(kf_spread_s.dropna())

print(f"{'Spread type':<28} {'ADF statistic':>14}  {'p-value':>10}  {'Stationary?':>12}")
print("-" * 68)
print(f"{'Static OLS (fixed β)':<28} {adf_static[0]:>14.4f}  {adf_static[1]:>10.6f}  {'Yes' if adf_static[1]<0.05 else 'No':>12}")
print(f"{'Kalman Filter (adaptive β)':<28} {adf_kf[0]:>14.4f}  {adf_kf[1]:>10.6f}  {'Yes' if adf_kf[1]<0.05 else 'No':>12}")

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(static_spread, color='steelblue', linewidth=0.7, alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[0].set_title(f'Static OLS Spread  (ADF p = {adf_static[1]:.5f})')
axes[0].set_ylabel('Spread (INR)')
axes[0].grid(alpha=0.2)

axes[1].plot(kf_spread_s, color='crimson', linewidth=0.7, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.5, linestyle='--')
axes[1].set_title(f'Kalman Adaptive Spread  (ADF p = {adf_kf[1]:.5f})')
axes[1].set_ylabel('Innovation e_t')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## Section C: Adaptive Backtest with Kalman Spread

The backtest uses:
- **Kalman spread** e_t as the signal series  
- **Rolling Z-score** of e_t (252-day lookback — same protocol as NB06 for fair comparison)  
- **Identical entry/exit/TC parameters** as NB06

This isolates the contribution of the adaptive hedge ratio, controlling for all other factors.

In [ ]:
ROLL     = 252
ENTRY_Z  = 2.0
EXIT_Z   = 0.5
TC       = 10 / 10_000   # 10 bps one-way

# ── Rolling Z-score of Kalman innovations ───────────────────────────────────
kf_roll_mean = kf_spread.rolling(ROLL, min_periods=60).mean()
kf_roll_std  = kf_spread.rolling(ROLL, min_periods=60).std()
kf_zscore    = (kf_spread - kf_roll_mean) / kf_roll_std

# ── Kalman spread returns (% of mean absolute level) ────────────────────────
kf_spread_mean_abs = kf_spread.dropna().abs().mean()
kf_spread_ret_pct  = kf_spread.diff() / kf_spread_mean_abs

# ── Baseline spread returns (from NB06 static β) for comparison ─────────────
static_spread_full  = A - beta_ols * B - alpha_ols
static_spread_ret   = static_spread_full.diff() / static_spread_full.dropna().abs().mean()
static_zscore       = pd.read_csv('data/zscore_tatasteel_hindalco.csv',
                                   index_col=0, parse_dates=True).iloc[:, 0]

print(f"Kalman Z-score valid rows: {kf_zscore.dropna().shape[0]}")

In [ ]:
def run_backtest(zscore, spread_ret_pct, tc=TC):
    """Vectorised backtest: returns position series and daily % P&L."""
    pos  = pd.Series(0.0, index=zscore.index)
    curr = 0.0

    for t in zscore.index:
        z = zscore.loc[t]
        if pd.isna(z):
            pos.loc[t] = 0.0
            continue
        if curr == 0:
            if   z >  ENTRY_Z: curr = -1.0
            elif z < -ENTRY_Z: curr =  1.0
        elif curr ==  1 and z > -EXIT_Z:  curr = 0.0
        elif curr == -1 and z <  EXIT_Z:  curr = 0.0
        pos.loc[t] = curr

    trade = pos.diff().abs().fillna(0)
    pnl   = pos.shift(1) * spread_ret_pct - trade * tc
    return pos, pnl.fillna(0)

pos_static, pnl_static = run_backtest(static_zscore, static_spread_ret)
pos_kalman, pnl_kalman = run_backtest(kf_zscore,     kf_spread_ret_pct)

cum_static = (1 + pnl_static).cumprod() - 1
cum_kalman = (1 + pnl_kalman).cumprod() - 1

print("Cumulative returns:")
print(f"  Static OLS (NB06 baseline) : {cum_static.iloc[-1]*100:+.2f}%")
print(f"  Kalman Filter (adaptive)   : {cum_kalman.iloc[-1]*100:+.2f}%")

In [ ]:
fig = plt.figure(figsize=(13, 9))
gs  = gridspec.GridSpec(3, 1, height_ratios=[3, 1.2, 1.2], hspace=0.4)

# ── Equity curves ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.plot(cum_static * 100, color='steelblue',  linewidth=1.2, label='Static OLS hedge (NB06 baseline)')
ax1.plot(cum_kalman * 100, color='crimson',    linewidth=1.5, label='Kalman Filter (adaptive hedge)')
ax1.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax1.set_ylabel('Cumulative Return (%)')
ax1.set_title('Adaptive (Kalman) vs Static Hedge Ratio — Equity Curves')
ax1.legend()
ax1.grid(alpha=0.25)

# ── Kalman β evolution ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.plot(kf_beta, color='darkorange', linewidth=0.8, label='Kalman β_t')
ax2.axhline(beta_ols, color='navy', linewidth=1.0, linestyle='--', label=f'OLS β={beta_ols:.3f}')
ax2.set_ylabel('Hedge Ratio β_t')
ax2.set_title('Time-Varying Kalman Hedge Ratio')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.2)

# ── Drawdown comparison ───────────────────────────────────────────────────
def max_drawdown_series(pnl):
    cum  = (1 + pnl).cumprod()
    peak = cum.cummax()
    return (cum - peak) / peak * 100

ax3 = fig.add_subplot(gs[2])
ax3.plot(max_drawdown_series(pnl_static), color='steelblue', linewidth=0.7, label='Static')
ax3.plot(max_drawdown_series(pnl_kalman), color='crimson',   linewidth=0.7, label='Kalman', alpha=0.8)
ax3.fill_between(max_drawdown_series(pnl_kalman).index,
                 max_drawdown_series(pnl_kalman), 0, alpha=0.1, color='crimson')
ax3.set_ylabel('Drawdown (%)')
ax3.set_title('Drawdown Profile')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.2)

plt.savefig('kalman_vs_static_equity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def report(pnl, pos, label):
    r       = pnl[pnl != 0]
    ann_ret = r.mean() * 252
    ann_vol = r.std()  * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum     = (1 + pnl).cumprod()
    dd      = (cum - cum.cummax()) / cum.cummax()
    max_dd  = dd.min()
    calmar  = ann_ret / abs(max_dd) if max_dd != 0 else np.nan
    hit     = (r > 0).mean()
    trades  = int((pos.diff().abs() > 0).sum() // 2)
    return pd.Series({
        'Ann. Return (%)': round(ann_ret * 100, 2),
        'Ann. Vol (%)':    round(ann_vol * 100, 2),
        'Sharpe':          round(sharpe,         3),
        'Max DD (%)':      round(max_dd * 100,   2),
        'Calmar':          round(calmar,          3),
        'Hit Rate (%)':    round(hit * 100,       1),
        'Trades':          trades,
    }, name=label)

summary = pd.DataFrame([
    report(pnl_static, pos_static, 'Static OLS Hedge'),
    report(pnl_kalman, pos_kalman, 'Kalman Adaptive Hedge'),
]).T

print("\n" + "="*55)
print("  FINAL COMPARISON: Static OLS vs Kalman Filter")
print("="*55)
print(summary.to_string())
print("="*55)

In [ ]:
kf_results = pd.DataFrame({
    'kf_alpha':  kf_alpha,
    'kf_beta':   kf_beta,
    'kf_spread': kf_spread,
    'kf_zscore': kf_zscore,
})
kf_results.to_csv('data/kalman_adaptive_hedge.csv')
print("Saved: data/kalman_adaptive_hedge.csv")

## Final Conclusion

### Why the Kalman Filter earns the "Adaptive" in the project title

| Property | Static OLS | Kalman Filter |
|----------|-----------|---------------|
| Hedge ratio | Constant (β = 0.245, fitted on full sample) | Time-varying β_t updated at every observation |
| Look-ahead | Yes (uses 10-year data to set β at day 1) | No (uses only data up to each date) |
| Regime adaptation | None | Tracks structural shifts in TATASTEEL/HINDALCO ratio |
| Spread stationarity | ADF-confirmed if cointegration holds | Innovation sequence is more robustly stationary |

### Design choices

- **δ = 1e-4 (process noise):** chosen to balance responsiveness vs noise.  
  Higher δ adapts faster but trades more; lower δ is more stable but may lag structural breaks.  
- **Online R estimation:** observation noise R is updated via exponential smoothing of
  squared innovations, avoiding the need to pre-specify it.  
- **No external library required:** the filter is implemented from first principles in NumPy,
  making it transparent and auditable.

### Project architecture summary

```
NB00 → Pair selection (ADF on OLS residual, not raw diff)
NB01 → Data download (NIFTY 100, 10 years)
NB02 → Cointegration validation
NB03 → Spread construction (with α) + rolling Z-score (no look-ahead)
NB03A→ Diagnostics (ARCH test justifies GARCH)
NB04 → GARCH(1,1)-t volatility modeling
NB05 → ML signals (RF, fixed RF bug, real probabilities saved)
NB06 → Backtest: baseline vs ML-filtered vs vol-scaled (real TC, % returns)
NB07 → Kalman Filter adaptive hedge ratio (THIS NOTEBOOK)
```